<a href="https://colab.research.google.com/github/ivanduzunov/AI-Agents-and-Workflows-for-Developers/blob/main/Multi_Agent_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install q langchain langchain-openai

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.types import Send, interrupt, Command
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Literal, List, Annotated, Dict
from IPython.display import Image
from google.colab import userdata

openai_key = userdata.get('OPENAI_KEY')

In [ ]:
def merge_dicts(a: Dict, b: Dict) -> Dict:
  return {**a, **b}

def merge_lists(a: list[str], b: list[str]) -> List:
  return [*a, *b]

class EditorReviewState(TypedDict):
    verdict: Literal["approve", "reject"]
    comments: List[str]

class MarketingState(TypedDict):
    product_brief: str
    target_platform: str
    research: str
    draft: str
    review: EditorReviewState
    revision_cycles: int

class SystemState(TypedDict):
    product_briefs: List[str]
    target_platforms: List[str]
    outcome: Annotated[Dict[str, str], merge_dicts]

In [ ]:
trend_analyst_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_key, reasoning_effort="low")
trend_analyst_prompt = """You are the Trend Analyst on a social media campaign team.

ROLE
Your job is to produce a concise research brief that can be used to draft posts.
You do NOT write marketing copy yourself — your output is intelligence, not creative.

PLATFORM CONTEXT
You are researching specifically for {platform}.
All research dimensions below must be filtered through this platform lens.
A pain point or a hashtag strategy that works on one platform may be wrong for other. Tailor everything to {platform}.

INPUT
You will receive a product brief describing what is being launched and the intended audience.

TASK
Analyze the brief and produce research covering:
1. Audience profile — who they are beyond the surface description (demographics, psychographics, where they spend time online, what they distrust)
2. Pain points — 3-5 concrete problems this audience has that the product plausibly addresses
3. Cultural trends — 2-4 current movements, conversations, or aesthetics in this niche the campaign could ride
4. Hashtags — a tiered list: 2-3 broad/high-volume, 3-5 mid-tier niche, 2-3 community-specific
5. Tone signals — what voice resonates with this audience (e.g., "dry humor over earnestness", "data over hype")
6. Format guidance — what post format performs best on {platform} for this kind of campaign (e.g., carousel vs. single image, thread vs. standalone, short-form video vs. static). Include length expectations.
7. Things to avoid — clichés, overused phrases, or angles that have been done to death in this space

CONSTRAINTS
- Your research MUST be based on plausible facts about the market. Do not invent statistics or cite specific sources you cannot verify.
- Stay neutral and analytical. No marketing voice, no hype words, no exclamation marks.
"""


editor_model = ChatOpenAI(model="gpt-5.4-mini", api_key=openai_key, reasoning_effort="low").with_structured_output(EditorReviewState)
editor_prompt = """You are the Editor on a social media campaign team.

ROLE
You are the quality gate. You critique the Copywriter's draft against a fixed rubric and decide whether it ships or goes back for revision. You are skeptical by default — "fine" is not "approved".

PLATFORM CONTEXT
You are evaluating exclusively against the norms of {platform}.
Your job is not just "is this good writing" — it is "is this the right post for {platform}".

INPUT
You will receive:
- Product brief: the original request
- Research: the Trend Analyst's findings (use this as ground truth for tone and audience)
- Draft: the Copywriter's current draft
- Count of revisions: how many times this draft has cycled (use this to calibrate, not to lower standards)

EVALUATION RUBRIC
Score the draft on each dimension. Each must pass for overall approval.

1. Tone match — Does the voice align with the tone signals in the research?
2. Audience fit — Does it speak to the identified audience and at least one stated pain point?
3. Cliché-free — Is it free of generic marketing language? Flag specific phrases.
4. Hook strength — Does the first line earn the second line?
5. CTA clarity — Is there one unambiguous action the reader can take?
6. Format hygiene — Hashtag count and placement, length appropriate to platform, no broken punctuation.

DECISION
- APPROVE only if every dimension passes. Approval means this is publishable as-is.
- REJECT if any dimension fails. Rejection must come with specific, actionable feedback — not "make it better" but "the hook 'Stay hydrated, stay winning' is a cliché; rewrite around the post-workout recovery angle from the research".

OUTPUT FORMAT
Return JSON with:
- verdict: "approve" or "reject"
- scores: object with each rubric dimension as a key, value is "pass" or "fail"
- revision_notes: array of specific change requests (empty if approved). Each note should reference the exact phrase or issue and suggest a direction, not a rewrite.

CONSTRAINTS
- You do not rewrite the copy yourself. Your job is to diagnose, not to draft.
- Be specific. "Tone is off" is not feedback; "The phrase 'unleash your potential' clashes with the dry, data-driven tone the Analyst identified" is feedback.
- Do not lower the bar on later revision cycles. If the draft still fails on cycle 3, it still fails. Loop termination is the orchestrator's job, not yours.
"""

In [ ]:
def empty_fn(state: MarketingState):
  pass

def analyst(state: MarketingState):
  product_brief = state.get("product_brief", "")
  target_platform = state.get("target_platform", "<none>")
  response = trend_analyst_model.invoke(
      input=[
          SystemMessage(PromptTemplate.from_template(trend_analyst_prompt).format(platform=target_platform)),
          HumanMessage(product_brief)
      ]
  )
  # todo: AI
  return {"research": response.text}

def writer(state: MarketingState):
  # todo: AI
  return {"draft": "The writer works..."}

def editor(state: MarketingState):
  draft = state.get("draft", "")
  target_platform = state.get("target_platform", "")
  revision_cycles = state.get("revision_cycles", 0)
  if revision_cycles >= 3:
    decision = interrupt({"draft": draft, "target_platform": target_platform})
    return {"review": decision}
  return {"review": {"verdict": "reject", "comments": []}, "revision_cycles": revision_cycles + 1}

def editor_path(state: MarketingState):
  review = state.get("review", {})
  verdict = review.get("verdict")
  if verdict == "approve":
    return END
  else:
    return "Writer"

In [ ]:
marketing_graph_builder = StateGraph(MarketingState)
marketing_graph_builder.add_node("Analyst", analyst)
marketing_graph_builder.add_node("Writer", writer)
marketing_graph_builder.add_node("Editor", editor)

marketing_graph_builder.add_edge(START, "Analyst")
marketing_graph_builder.add_edge("Analyst", "Writer")
marketing_graph_builder.add_edge("Writer", "Editor")
marketing_graph_builder.add_conditional_edges("Editor", editor_path, ["Writer", END])

marketing_graph = marketing_graph_builder.compile()

In [ ]:
display(Image(marketing_graph.get_graph(xray=True).draw_mermaid_png(), format="png"))

In [ ]:
def supervisor(state: SystemState):
  pass

def produce_outcome(state: MarketingState):
  target_platform = state.get("target_platform")
  final_post = state.get("draft")
  return {"outcome": {target_platform: final_post}}

def delegate_tasks(state: SystemState):
  product_briefs = state.get("product_briefs", [])
  target_platforms = state.get("target_platforms", [])

  tasks = [Send("Marketing Team", {"product_brief": pb, "target_platform": tb}) for tb in target_platforms for pb in product_briefs]

  if not tasks:
    return END
  return tasks

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
system_graph_buildr = StateGraph(SystemState)
system_graph_buildr.add_node("Supervisor", supervisor)
system_graph_buildr.add_node("Marketing Team", marketing_graph | produce_outcome)

system_graph_buildr.add_edge(START, "Supervisor")
system_graph_buildr.add_conditional_edges("Supervisor", delegate_tasks, ["Marketing Team", END])
system_graph_buildr.add_edge("Marketing Team", END)

system_graph = system_graph_buildr.compile(checkpointer=checkpointer, debug=True)

In [ ]:
display(Image(system_graph.get_graph(xray=True).draw_mermaid_png(), format="png"))

In [ ]:
marketing_graph.invoke(input={"product_brief": "We are launching a new eco-friendly smart water bottle. Target: gym-goers.", "target_platform": "LinkedIn"})

In [ ]:
tread1_config = {"configurable": {"thread_id": "tr_1"}}
state1 = system_graph.invoke(input={"product_briefs": ["We are launching a new eco-friendly smart water bottle. Target platform: LinkedIn."], "target_platforms": ["Linkedin", "Facebook"]},
                           config=tread1_config)


In [ ]:
state2 = system_graph.invoke(input=Command(resume={i.id: {"verdict": "approve"} for i in state1["__interrupt__"]}),
                           config=tread1_config)

In [ ]:
# EXAMPLE PART

In [ ]:
class SystemState(TypedDict):
    pass

def main(state: SystemState):
  print("Executing node MAIN")

def a(state: SystemState):
  print("Executing node A")

def b1(state: SystemState):
  print("Executing node B1")

def b2(state: SystemState):
  print("Executing node B2")

def finish(state: SystemState):
  print("Executing node FINISH")

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
sub_graph_builder = StateGraph(SystemState)
sub_graph_builder.add_node("B1", b1)
sub_graph_builder.add_node("B2", b2)

sub_graph_builder.add_edge(START, "B1")
sub_graph_builder.add_edge("B1", "B2")
sub_graph_builder.add_edge("B2", END)

sub_graph = sub_graph_builder.compile()

In [ ]:
graph_builder = StateGraph(SystemState)
graph_builder.add_node("main", main)
graph_builder.add_node("A", a)
graph_builder.add_node("B", sub_graph)
graph_builder.add_node("finish", finish)


graph_builder.add_edge(START, "main")
graph_builder.add_edge("main", "A")
graph_builder.add_edge("main", "B")
graph_builder.add_edge("A", "finish")
graph_builder.add_edge("B", "finish")


graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
display(Image(graph.get_graph(xray=True).draw_mermaid_png(), format="png"))

In [ ]:
thread1_config = {"configurable": {"thread_id": "tr_1"}}
graph.invoke(input={}, config=thread1_config)

In [ ]:
list(checkpointer.list(thread1_config))

In [ ]:
list(graph.get_state_history(thread1_config))